# Assignment 1: Introduction to language modeling

Compact solution notebook. Tasks are separated, and 🎓 tasks include a short why/how note.

## ⚙ Task 0.1: Setting up the environment

In [1]:
from pathlib import Path
from collections import Counter
import math
import random
import pickle

import nltk
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel, TrainingArguments
from transformers.modeling_outputs import CausalLMOutput

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


def get_device(use_cpu=False):
    if use_cpu:
        return torch.device('cpu')
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print('device:', device)


/Users/telio/miniconda3/envs/phenoVLM-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## ⚙ Task 1.1: Using NLTK or SpaCy for word splitting

In [2]:
A1_DATA_URL = 'https://www.cse.chalmers.se/~richajo/waspnlp2026/a1_1.zip'
DATA_ROOT = Path('data')
DATA_DIR = DATA_ROOT / 'a1_1'
ARCHIVE_FILE = DATA_ROOT / 'a1_1.zip'
TRAIN_FILE = DATA_DIR / 'train.txt'
VAL_FILE = DATA_DIR / 'val.txt'


def ensure_assignment_data():
    if TRAIN_FILE.exists() and VAL_FILE.exists():
        return

    from urllib.request import urlretrieve
    import zipfile

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print('Downloading Assignment 1 data...')
    urlretrieve(A1_DATA_URL, ARCHIVE_FILE)
    print('Extracting Assignment 1 data...')
    with zipfile.ZipFile(ARCHIVE_FILE) as archive:
        archive.extractall(DATA_ROOT)

    assert TRAIN_FILE.exists(), f'Missing expected file after extraction: {TRAIN_FILE}'
    assert VAL_FILE.exists(), f'Missing expected file after extraction: {VAL_FILE}'


ensure_assignment_data()


def lowercase_tokenizer(text):
    return [token.lower() for token in nltk.word_tokenize(text)]


print('data files:', TRAIN_FILE, VAL_FILE)
print(lowercase_tokenizer("Let's test!!"))


data files: data/a1_1/train.txt data/a1_1/val.txt
['let', "'s", 'test', '!', '!']


## 🎓 Task 1.2: Building the vocabulary

**Why/how.** A language model predicts token IDs, so every word must map to an integer. The `max_voc_size` hyperparameter caps the vocabulary; after reserving the four special symbols, we keep the most frequent training tokens and map the rest to `<UNK>`.


In [3]:
MAX_VOC_SIZE = 20_000
MODEL_MAX_LENGTH = 80
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'


def read_nonempty_lines(path):
    with path.open(encoding='utf-8') as handle:
        return [line.strip() for line in handle if line.strip()]


train_texts = read_nonempty_lines(TRAIN_FILE)
val_texts = read_nonempty_lines(VAL_FILE)
tokenized_train = [lowercase_tokenizer(text) for text in train_texts]
tokenized_val = [lowercase_tokenizer(text) for text in val_texts]


def build_vocabulary(tokenized_texts, max_voc_size=None):
    special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]
    if max_voc_size is not None and max_voc_size < len(special_tokens):
        raise ValueError('max_voc_size must leave room for the four special tokens')

    counts = Counter(token for text in tokenized_texts for token in text)
    max_words = None if max_voc_size is None else max_voc_size - len(special_tokens)
    words = [word for word, _ in counts.most_common(max_words) if word not in special_tokens]

    int_to_str = special_tokens + words
    str_to_int = {word: idx for idx, word in enumerate(int_to_str)}
    return str_to_int, int_to_str, counts


str_to_int, int_to_str, token_counts = build_vocabulary(tokenized_train, max_voc_size=MAX_VOC_SIZE)
print('train paragraphs:', len(train_texts))
print('validation paragraphs:', len(val_texts))
print('vocab size:', len(str_to_int))
print('special ids:', {tok: str_to_int[tok] for tok in [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]})
print('contains frequent words:', {'the': 'the' in str_to_int, 'and': 'and' in str_to_int})
print('contains rare examples:', {'cuboidal': 'cuboidal' in str_to_int, 'epiglottis': 'epiglottis' in str_to_int})
assert len(str_to_int) <= MAX_VOC_SIZE


train paragraphs: 147059
validation paragraphs: 17874
vocab size: 20000
special ids: {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
contains frequent words: {'the': True, 'and': True}
contains rare examples: {'cuboidal': False, 'epiglottis': False}


## ⚙ Task 1.3: Implementing a HuggingFace-like Tokenizer

In [4]:
class A1Tokenizer:
    """A minimal tokenizer with a HuggingFace-like call interface."""

    def __init__(self, str_to_int, int_to_str, tokenize_fun=lowercase_tokenizer, model_max_length=None):
        self.str_to_int = str_to_int
        self.int_to_str = int_to_str
        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length
        self.pad_token_id = str_to_int[PAD_TOKEN]
        self.unk_token_id = str_to_int[UNK_TOKEN]
        self.bos_token_id = str_to_int[BOS_TOKEN]
        self.eos_token_id = str_to_int[EOS_TOKEN]

    def __len__(self):
        return len(self.int_to_str)

    def encode(self, text, truncation=False):
        tokens = [BOS_TOKEN] + self.tokenize_fun(text) + [EOS_TOKEN]
        ids = [self.str_to_int.get(token, self.unk_token_id) for token in tokens]
        if truncation and self.model_max_length is not None and len(ids) > self.model_max_length:
            ids = ids[:self.model_max_length]
            ids[-1] = self.eos_token_id
        return ids

    def decode(self, ids, skip_special_tokens=True):
        words = []
        for idx in ids:
            word = self.int_to_str[int(idx)] if int(idx) < len(self.int_to_str) else UNK_TOKEN
            if skip_special_tokens and word in {PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN}:
                continue
            words.append(word)
        return ' '.join(words)

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        if isinstance(texts, str):
            texts = [texts]
        if return_tensors not in {None, 'pt'}:
            raise ValueError("return_tensors must be None or 'pt'")

        encoded = [self.encode(text, truncation=truncation) for text in texts]
        attention_mask = [[1] * len(ids) for ids in encoded]

        if padding:
            max_len = max(len(ids) for ids in encoded)
            encoded = [ids + [self.pad_token_id] * (max_len - len(ids)) for ids in encoded]
            attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

        if return_tensors == 'pt':
            encoded = torch.tensor(encoded, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        return BatchEncoding({'input_ids': encoded, 'attention_mask': attention_mask})

    def save(self, filename):
        with open(filename, 'wb') as handle:
            pickle.dump(self, handle)

    @staticmethod
    def from_file(filename):
        with open(filename, 'rb') as handle:
            return pickle.load(handle)


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None):
    texts = read_nonempty_lines(Path(train_file))
    tokenized = [tokenize_fun(text) for text in texts]
    str_to_int, int_to_str, _ = build_vocabulary(tokenized, max_voc_size=max_voc_size)
    return A1Tokenizer(str_to_int, int_to_str, tokenize_fun=tokenize_fun, model_max_length=model_max_length)


tokenizer = build_tokenizer(TRAIN_FILE, max_voc_size=MAX_VOC_SIZE, model_max_length=MODEL_MAX_LENGTH)
test_texts = ['This is a test.', 'Another test.']
print(tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True))
print(tokenizer.decode(tokenizer.encode('She lives in San')))


{'input_ids': tensor([[  2,  35,  14,  11, 975,   6,   3],
        [  2, 155, 975,   6,   3,   0,   0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])}
she lives in san


## ⚙ Task 2.1: Loading the texts

In [5]:
HF_DATASETS_CACHE = Path('data/hf_datasets_cache')
HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)

dataset = load_dataset(
    'text',
    data_files={'train': str(TRAIN_FILE), 'val': str(VAL_FILE)},
    cache_dir=str(HF_DATASETS_CACHE),
)
dataset = dataset.filter(lambda example: example['text'].strip() != '')
print(dataset)

USE_SMALL_DATASET = False
N_TRAIN_EXAMPLES = 1_000
N_VAL_EXAMPLES = 1_000

if USE_SMALL_DATASET:
    dataset['train'] = Subset(dataset['train'], range(min(N_TRAIN_EXAMPLES, len(dataset['train']))))
    dataset['val'] = Subset(dataset['val'], range(min(N_VAL_EXAMPLES, len(dataset['val']))))

print('used train examples:', len(dataset['train']))
print('used validation examples:', len(dataset['val']))


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 147059
    })
    val: Dataset({
        features: ['text'],
        num_rows: 17874
    })
})
used train examples: 147059
used validation examples: 17874


## ⚙ Task 2.2: Iterating through the datasets

In [6]:
def collate_texts(batch):
    return [example['text'] for example in batch]


train_dl = DataLoader(dataset['train'], batch_size=4, shuffle=True, collate_fn=collate_texts)
valid_dl = DataLoader(dataset['val'], batch_size=4, collate_fn=collate_texts)

batch_texts = next(iter(train_dl))
encoded = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True)
print(batch_texts[0][:120])
print(encoded['input_ids'].shape)
print(encoded['attention_mask'].shape)


Zebu cattle (Bos Indicus), introduced to Brazil in the last century, were extensively crossbred with herds of native cat
torch.Size([4, 80])
torch.Size([4, 80])


## 🎓 Task 3.1: Setting up the network

**Why/how.** The model embeds tokens, processes the sequence with a GRU, and projects each hidden state back to vocabulary logits. The output shape is `(batch, time, vocab_size)` because we predict one next-token distribution per position.

In [7]:
class A1RNNModelConfig(PretrainedConfig):
    model_type = 'a1-rnn-lm'

    def __init__(self, vocab_size=0, embedding_size=64, hidden_size=96, num_layers=1, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout


class A1RNNModel(PreTrainedModel):
    config_class = A1RNNModelConfig

    def __init__(self, config):
        super().__init__(config)
        self.embedding = nn.Embedding(config.vocab_size, config.embedding_size)
        self.rnn = nn.GRU(
            config.embedding_size,
            config.hidden_size,
            num_layers=config.num_layers,
            batch_first=True,
            dropout=config.dropout if config.num_layers > 1 else 0.0,
        )
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size)
        self.loss_func = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, labels=None):
        embedded = self.embedding(input_ids)
        rnn_out, _ = self.rnn(embedded)
        logits = self.unembedding(rnn_out)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = self.loss_func(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return CausalLMOutput(loss=loss, logits=logits)


config = A1RNNModelConfig(vocab_size=len(tokenizer))
model = A1RNNModel(config).to(device)
out = model(encoded['input_ids'].to(device))
print(out.logits.shape)


torch.Size([4, 80, 20000])


## 🎓 Task 3.2: Computing the loss

**Why/how.** Language modeling trains by comparing the logits at position `t` to the true token at position `t+1`. Flattening batch and time lets `cross_entropy` treat every position as one classification example.

In [8]:
def make_lm_batch(texts, tokenizer, device):
    encoded = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    input_ids = encoded['input_ids'].to(device)
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return input_ids, labels


input_ids, labels = make_lm_batch(batch_texts, tokenizer, device)
out = model(input_ids=input_ids, labels=labels)
print('loss:', float(out.loss))


loss: 9.943944931030273


## 🎓 Task 4.1: Implementing the trainer

**Why/how.** Each step computes the next-token loss, backpropagates, clips gradients for stability, and updates parameters with AdamW. Validation loss is converted to perplexity for interpretability.

The training settings are collected at the top of the next cell. Increase `TRAIN_EPOCHS` for a longer run and change `TRAIN_BATCH_SIZE` / `EVAL_BATCH_SIZE` to test throughput and stability. Rerun Task 3.1 first if you want to restart from a freshly initialized model.


In [9]:
class A1Trainer:
    """A small Trainer-like class for this assignment."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer
        self.device = get_device(use_cpu=args.use_cpu)

    def _loader(self, dataset, batch_size, shuffle=False):
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_texts)

    def evaluate(self):
        self.model.eval()
        loader = self._loader(self.eval_dataset, self.args.per_device_eval_batch_size)
        losses = []
        with torch.no_grad():
            for texts in loader:
                input_ids, labels = make_lm_batch(texts, self.tokenizer, self.device)
                losses.append(self.model(input_ids=input_ids, labels=labels).loss.item())
        return sum(losses) / max(1, len(losses))

    def train(self):
        self.model.to(self.device)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.args.learning_rate)
        train_loader = self._loader(self.train_dataset, self.args.per_device_train_batch_size, shuffle=True)
        history = []

        for epoch in range(1, int(self.args.num_train_epochs) + 1):
            self.model.train()
            total = 0.0
            for texts in train_loader:
                input_ids, labels = make_lm_batch(texts, self.tokenizer, self.device)
                loss = self.model(input_ids=input_ids, labels=labels).loss
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                total += loss.item()

            train_loss = total / max(1, len(train_loader))
            valid_loss = self.evaluate() if self.args.eval_strategy == 'epoch' else float('nan')
            row = {
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'valid_ppl': math.exp(min(valid_loss, 20)),
            }
            history.append(row)
            print(row)

        print(f'Saving to {self.args.output_dir}.')
        self.model.save_pretrained(self.args.output_dir)
        return history


# Training knobs. For the full dataset, set USE_SMALL_DATASET = False in Task 2.1.
TRAIN_EPOCHS = 5
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 3e-3
USE_CPU = False
OUTPUT_DIR = f'results/assignments/a1_rnn_epochs{TRAIN_EPOCHS}_bs{TRAIN_BATCH_SIZE}'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    optim='adamw_torch',
    eval_strategy='epoch',
    use_cpu=USE_CPU,
    learning_rate=LEARNING_RATE,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to=[],
    save_strategy='no',
)

print({
    'epochs': TRAIN_EPOCHS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'output_dir': OUTPUT_DIR,
})

trainer = A1Trainer(model, training_args, dataset['train'], dataset['val'], tokenizer)
history = trainer.train()


{'epochs': 5, 'train_batch_size': 32, 'eval_batch_size': 64, 'learning_rate': 0.003, 'output_dir': 'results/assignments/a1_rnn_epochs5_bs32'}
{'epoch': 1, 'train_loss': 5.408083802414937, 'valid_loss': 4.999028858116695, 'valid_ppl': 148.26909883054256}
{'epoch': 2, 'train_loss': 4.87089661248774, 'valid_loss': 4.832114957060132, 'valid_ppl': 125.47605669456293}
{'epoch': 3, 'train_loss': 4.702459184370215, 'valid_loss': 4.75846472467695, 'valid_ppl': 116.56682627868344}
{'epoch': 4, 'train_loss': 4.605500414850818, 'valid_loss': 4.709411520617349, 'valid_ppl': 110.98682722454913}
{'epoch': 5, 'train_loss': 4.541678039378141, 'valid_loss': 4.680527259622301, 'valid_ppl': 107.82691036197909}
Saving to results/assignments/a1_rnn_epochs5_bs32.


## ⚙ Task 5.1: Predicting the next word

In [13]:
def predict_next(model, prompt, topk=5):
    model.eval()
    encoded = tokenizer([prompt], return_tensors='pt', padding=False, truncation=True)
    input_ids = encoded['input_ids'].to(trainer.device)
    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -2]
        probs = logits.softmax(dim=-1)
        values, indices = probs.topk(topk)
    return [(tokenizer.int_to_str[int(idx)], float(prob)) for idx, prob in zip(indices, values)]


predict_next(model, 'She lives in San')


[('francisco', 0.5076640248298645),
 ('<UNK>', 0.0988302156329155),
 ('marco', 0.05192309990525246),
 ('jose', 0.04012627527117729),
 ('diego', 0.030872782692313194)]

## 🎓 Task 5.2: Computing the perplexity

**Why/how.** Perplexity is `exp(cross_entropy)`. It can be read as the model's effective average branching factor: lower is better.

In [14]:
def perplexity(loss):
    return math.exp(min(loss, 20))


valid_loss = trainer.evaluate()
print('valid loss:', valid_loss)
print('valid perplexity:', perplexity(valid_loss))


valid loss: 4.680527259622301
valid perplexity: 107.82691036197909


## 🎓 Task 5.3: Inspecting the learned word embeddings

**Why/how.** The embedding matrix stores one vector per word. Cosine similarity gives a simple way to inspect which words the model has placed near each other.

In [15]:
def nearest_words(model, word, n_neighbors=5):
    if word not in tokenizer.str_to_int:
        raise ValueError(f'{word!r} is not in the vocabulary')

    emb = model.embedding.weight.detach().cpu()
    word_id = tokenizer.str_to_int[word]
    scores = F.cosine_similarity(emb[word_id][None, :], emb, dim=1)
    values, indices = scores.topk(n_neighbors + 1)
    neighbors = []
    for idx, value in zip(indices.tolist(), values.tolist()):
        token = tokenizer.int_to_str[idx]
        if token != word and token not in {PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN}:
            neighbors.append((token, value))
    return neighbors[:n_neighbors]


nearest_words(model, 'sweden')


[('germany', 0.5385755300521851),
 ('france', 0.5229812264442444),
 ('europe', 0.5147715210914612),
 ('hungary', 0.5128011107444763),
 ('luxembourg', 0.508007287979126)]